In [1]:
#Load and inspect dataset

In [2]:
import pandas as pd

df = pd.read_csv("promoters.data", header=None)

df.columns = ["Label", "ID", "Sequence"]

print(df.head())
print("\nDataset shape:", df.shape)
print("\nClass distribution:\n", df["Label"].value_counts())

  Label         ID                                           Sequence
0     +        S10  \t\ttactagcaatacgcttgcgttcggtggttaagtatgtataat...
1     +       AMPC  \t\ttgctatcctgacagttgtcacgctgattggtgtcgttacaat...
2     +       AROH  \t\tgtactagagaactagtgcattagcttatttttttgttatcat...
3     +      DEOP2  \taattgtgatgtgtatcgaagtgtgttgcggagtagatgttagaa...
4     +  LEU1_TRNA  \ttcgataattaactattgacgaaaagctgaaaaccactagaatgc...

Dataset shape: (106, 3)

Class distribution:
 Label
+    53
-    53
Name: count, dtype: int64


In [3]:
#Clean Sequences Properly

In [4]:
# Remove tab characters and convert to lowercase

df["Sequence"] = df["Sequence"].str.replace("\t", "", regex=False)
df["Sequence"] = df["Sequence"].str.strip()
df["Sequence"] = df["Sequence"].str.lower()

print(df["Sequence"].head())
print("Length of first sequence:", len(df["Sequence"][0]))

0    tactagcaatacgcttgcgttcggtggttaagtatgtataatgcgc...
1    tgctatcctgacagttgtcacgctgattggtgtcgttacaatctaa...
2    gtactagagaactagtgcattagcttatttttttgttatcatgcta...
3    aattgtgatgtgtatcgaagtgtgttgcggagtagatgttagaata...
4    tcgataattaactattgacgaaaagctgaaaaccactagaatgcgc...
Name: Sequence, dtype: object
Length of first sequence: 57


In [5]:
#Generate 3-mer Features

In [6]:
from itertools import product
import numpy as np

# Generate all possible 3-mers
kmers = [''.join(p) for p in product('atcg', repeat=3)]

def kmer_count(sequence, k=3):
    counts = dict.fromkeys(kmers, 0)
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i+k]
        if kmer in counts:
            counts[kmer] += 1
    return list(counts.values())

# Apply to all sequences
kmer_features = df["Sequence"].apply(kmer_count)

# Convert to DataFrame
kmer_df = pd.DataFrame(
    np.vstack(kmer_features.values),
    columns=[f"kmer_{k}" for k in kmers]
)

print("K-mer feature shape:", kmer_df.shape)
kmer_df.head()

K-mer feature shape: (106, 64)


,kmer_aaa,kmer_aat,kmer_aac,kmer_aag,kmer_ata,kmer_att,kmer_atc,kmer_atg,kmer_aca,kmer_act,...,kmer_gtc,kmer_gtg,kmer_gca,kmer_gct,kmer_gcc,kmer_gcg,kmer_gga,kmer_ggt,kmer_ggc,kmer_ggg
0,0,2,0,1,2,0,0,2,0,1,...,1,1,1,2,0,3,0,2,1,1
1,0,1,1,0,0,1,3,0,2,0,...,2,1,1,2,1,0,0,1,0,0
2,0,0,2,0,0,2,1,1,0,2,...,0,1,1,2,0,1,0,0,1,0
3,1,2,2,1,1,1,1,2,1,2,...,0,4,0,0,0,1,1,0,0,0
4,4,2,2,1,1,2,0,1,0,2,...,0,1,0,1,1,1,0,1,0,0


In [7]:
#Encode Labels

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df["Label"])  # + and - to 1 and 0

print("Encoded labels:", le.classes_)

Encoded labels: ['+' '-']


In [9]:
#Train Random Forest Model

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X = kmer_df

# Train-test split (stratified because small dataset)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    random_state=42
)

model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

print("=== Promoter Classification Results ===")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

=== Promoter Classification Results ===
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        11
           1       1.00      0.91      0.95        11

    accuracy                           0.95        22
   macro avg       0.96      0.95      0.95        22
weighted avg       0.96      0.95      0.95        22

Accuracy: 0.9545454545454546


In [11]:
#Cross-Validation

In [12]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5
)

print("Cross-validation scores:", cv_scores)
print("Mean CV accuracy:", cv_scores.mean())

Cross-validation scores: [0.86363636 1.         0.95238095 0.9047619  0.95238095]
Mean CV accuracy: 0.9346320346320345
